# Giai đoạn 2 – Tự thu thập dữ liệu từ GOV.UK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23010216-hieusphamj/text-summarization-nlp/blob/main/notebooks/02_crawl_govuk.ipynb)

**Nguồn:** bài tin và thông cáo báo chí trên GOV.UK. Mỗi trang có trường `description` do biên tập viên viết, ta dùng làm **tóm tắt tham chiếu**, còn nội dung `details.body` là **bài gốc**.
**Giấy phép:** Open Government Licence v3.0 (xem `data/DATA_CARD.md`). Thu thập qua Content/Search API, nghỉ 0,5 giây giữa các request, có User-Agent định danh.

**Quy trình:** liệt kê bài theo (loại bài × tháng) → tải nội dung (có cache) → HTML sang văn bản → lọc chất lượng → khử trùng lặp → chia 80/10/10 → so sánh với CNN/DM.

> **Trước khi chạy:** mở `configs/data_govuk.yaml`, thay `THAY_EMAIL_CUA_BAN@example.com` trong `user_agent` bằng email thật của bạn rồi commit/push (hoặc sửa trực tiếp trên Colab).

In [ ]:
# 0. Thiết lập môi trường
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/23010216-hieusphamj/text-summarization-nlp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("src").exists() and not Path("text-summarization-nlp").exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
if Path("text-summarization-nlp").exists():
    os.chdir("text-summarization-nlp")
while not Path("src").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, os.getcwd())

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "beautifulsoup4", "requests"], check=True)
    # Lưu cache dữ liệu thô lên Google Drive để không mất khi Colab ngắt kết nối
    from google.colab import drive
    drive.mount("/content/drive")
    drive_raw = Path("/content/drive/MyDrive/nlp_summarization/govuk_raw"); drive_raw.mkdir(parents=True, exist_ok=True)
    raw = Path("data/raw/govuk"); raw.parent.mkdir(parents=True, exist_ok=True)
    if not raw.exists():
        raw.symlink_to(drive_raw, target_is_directory=True)
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# 1. Cấu hình chạy
CONFIG = "configs/data_govuk.yaml"
RUN_CRAWL = True      # False nếu đã crawl xong và chỉ muốn chạy lại phần làm sạch/EDA
MAX_DOCS = 300        # chạy thử 300 bài trước; khi ổn đổi thành None để dùng max_docs trong config (6000)

import yaml
cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
if RUN_CRAWL and "THAY_EMAIL" in cfg["user_agent"]:
    raise SystemExit("Hãy thay email thật vào user_agent trong configs/data_govuk.yaml rồi chạy lại.")

## 1. Kiểm tra nhanh API (smoke test)
In ra cấu trúc dữ liệu thật của vài bài để chắc các trường (`title`, `description`, `details.body`) đúng như kỳ vọng.

In [ ]:
from src.crawl.govuk_crawler import main as crawl
if RUN_CRAWL:
    crawl(CONFIG, do_smoke=True)

## 2. Thu thập dữ liệu thô
Cache từng bài vào `data/raw/govuk/` (trên Drive nếu dùng Colab). Có thể ngắt và chạy lại, bài đã tải sẽ được bỏ qua.

In [ ]:
if RUN_CRAWL:
    crawl(CONFIG, max_docs=MAX_DOCS)
print("Số bài thô:", len(list(Path(cfg["raw_dir"]).glob("*.json"))))

## 3. Làm sạch, lọc, khử trùng lặp, chia tập

In [ ]:
from src.crawl.build_dataset import main as build
splits = build(CONFIG)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import eda_utils as u
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
FIG = Path("results/figures"); FIG.mkdir(parents=True, exist_ok=True)
TAB = Path("results/tables");  TAB.mkdir(parents=True, exist_ok=True)

### 3.1 Phễu làm sạch (bao nhiêu bài bị loại ở mỗi bước)

In [ ]:
f = pd.read_csv(TAB / "govuk_cleaning_funnel.csv")
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(f["buoc"][::-1], f["con_lai"][::-1], color="#4a6fa5")
for i, v in enumerate(f["con_lai"][::-1]): ax.text(v, i, f" {v:,}", va="center")
ax.set_xlabel("số bài còn lại"); ax.set_title("Phễu làm sạch dữ liệu GOV.UK")
plt.tight_layout(); plt.savefig(FIG / "govuk_cleaning_funnel.png", dpi=200); plt.show()
f

## 4. EDA bộ dữ liệu tự thu thập

In [ ]:
d = {s: pd.read_json(f"data/processed/govuk_{s}.jsonl", lines=True) for s in ["train", "validation", "test"]}
tr = d["train"].copy()
print({k: len(v) for k, v in d.items()})
tr["art_words"] = tr["article"].map(u.word_count); tr["sum_words"] = tr["highlights"].map(u.word_count)
tr["sum_sents"] = tr["highlights"].map(lambda t: len(u.split_sentences(t)))
tr["compression"] = [u.compression_ratio(a, s) for a, s in zip(tr["article"], tr["highlights"])]
display(tr["doc_type"].value_counts().rename("số bài").to_frame())
display(pd.DataFrame({"article_words": u.describe(tr["art_words"]), "summary_words": u.describe(tr["sum_words"]),
                      "summary_sentences": u.describe(tr["sum_sents"])}).round(1))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].hist(tr["art_words"], bins=40, color="#4a6fa5"); ax[0].set_title("Độ dài bài (số từ)")
ax[1].hist(tr["sum_words"], bins=30, color="#3c8d53"); ax[1].set_title("Độ dài tóm tắt (số từ)")
ax[2].hist(tr["compression"].clip(upper=tr["compression"].quantile(0.99)), bins=40, color="#c9a227"); ax[2].set_title("Tỉ lệ nén")
for a in ax: a.set_ylabel("số mẫu")
plt.tight_layout(); plt.savefig(FIG / "govuk_length_hist.png", dpi=200); plt.show()

ym = pd.to_datetime(tr["date"], errors="coerce", utc=True).dt.to_period("Q").astype(str).value_counts().sort_index()
ym.plot(kind="bar", figsize=(9, 3), color="#4a6fa5", title="Số bài theo quý"); plt.tight_layout()
plt.savefig(FIG / "govuk_by_quarter.png", dpi=200); plt.show()

In [ ]:
N = min(3000, len(tr)); sub = tr.sample(N, random_state=42)
novel = {n: float(np.nanmean([u.novel_ngram_ratio(a, s, n) for a, s in zip(sub["article"], sub["highlights"])])) for n in (1, 2, 3)}
pos = np.array([p for a, h in zip(tr["article"].head(2000), tr["highlights"].head(2000)) for p in u.best_match_positions(a, h)])
within3, within10 = float((pos < 3).mean()), float((pos < 10).mean())
print({f"{k}-gram mới": f"{v*100:.1f}%" for k, v in novel.items()}, f"| 3 câu đầu: {within3*100:.1f}% | 10 câu đầu: {within10*100:.1f}%")

summary = pd.DataFrame([
    ("Số mẫu train/val/test", " / ".join(str(len(d[s])) for s in d)),
    ("Số từ bài – trung vị", round(tr["art_words"].median(), 1)),
    ("Số từ bài – p95", round(tr["art_words"].quantile(0.95), 1)),
    ("Số từ tóm tắt – trung vị", round(tr["sum_words"].median(), 1)),
    ("Số câu tóm tắt – trung bình", round(tr["sum_sents"].mean(), 2)),
    ("Tỉ lệ nén – trung vị (x)", round(tr["compression"].median(), 1)),
    ("% unigram mới", round(novel[1]*100, 1)), ("% bigram mới", round(novel[2]*100, 1)), ("% trigram mới", round(novel[3]*100, 1)),
    ("% câu tóm tắt có nguồn ở 3 câu đầu", round(within3*100, 1)),
    ("% câu tóm tắt có nguồn ở 10 câu đầu", round(within10*100, 1)),
], columns=["chỉ số", "GOV.UK"])
summary.to_csv(TAB / "govuk_eda_summary.csv", index=False)

### 4.1 So sánh với CNN/DailyMail (dữ liệu huấn luyện chính)

In [ ]:
cmp = summary.copy()
p = TAB / "cnn_dm_eda_summary.csv"
if p.exists():
    cnn = pd.read_csv(p).rename(columns={"giá trị": "CNN/DM"})
    cmp = cnn.merge(summary, on="chỉ số", how="right")[["chỉ số", "CNN/DM", "GOV.UK"]]
cmp.to_csv(TAB / "govuk_vs_cnn_dm.csv", index=False)
cmp

## 5. Xem ngẫu nhiên vài mẫu (kiểm tra chất lượng bằng mắt)

In [ ]:
for _, r in tr.sample(min(4, len(tr)), random_state=3).iterrows():
    print(f"[{r['doc_type']}] {r['title']}")
    print("  TÓM TẮT:", r["highlights"])
    print("  BÀI    :", r["article"][:300].replace("\n", " "), "...\n")

## 6. Nhận xét (điền sau khi chạy với MAX_DOCS đầy đủ, rồi chép vào `docs/decisions.md` mục D3)

- **Quy mô:** tải ... bài, sau làm sạch còn ... (train/val/test = ... / ... / ...). Bước loại nhiều nhất: ...
- **So với CNN/DM:** bài trung vị ... từ (CNN/DM: 637), tóm tắt trung vị ... từ (46), tỉ lệ nén ...x (13,6x).
- **Mức độ abstractive:** ...% bigram mới (CNN/DM: 52,2%).
- **Vị trí nguồn:** ...% trong 3 câu đầu (CNN/DM: 25,6%).
- **Hạn chế:** tóm tắt chỉ ... câu (kiểu một câu mô tả), nội dung là tin chính phủ Anh (miền hẹp).

In [ ]:
# 7. (Colab) Đóng gói kết quả để tải về và commit vào repo
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("phase2_outputs", "zip", ".", "results")
    shutil.copy("data/samples/govuk_sample.jsonl", "govuk_sample.jsonl")
    files.download("phase2_outputs.zip"); files.download("govuk_sample.jsonl")